## **Environment Setup**

In [1]:
!nvidia-smi

Sat Aug 22 14:58:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q ultralytics roboflow

In [3]:
import torch
import ultralytics

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Ultralytics:", ultralytics.__version__)

PyTorch: 2.11.0+cu128
CUDA available: True
Ultralytics: 8.4.126


## **Google Drive**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os

PROJECT_DIR = "/content/drive/MyDrive/drone_detection_project"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/videos", exist_ok=True)

print("Project directory:", PROJECT_DIR)

Project directory: /content/drive/MyDrive/drone_detection_project


## **Dataset**

In [6]:
from roboflow import Roboflow

rf = Roboflow(api_key="BZyoGQc5Y9XG6pzoOTis")
project = rf.workspace("uavs-7l7kv").project("uavs-vqpqt")
version = project.version(2)
dataset = version.download("yolov8")
print("Dataset location:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...
Dataset location: /content/UAVs-2


## **Dataset Verification**

In [7]:
from pathlib import Path
BASE = Path(dataset.location)
print("Dataset contents:")
print(list(BASE.iterdir()))

Dataset contents:
[PosixPath('/content/UAVs-2/README.dataset.txt'), PosixPath('/content/UAVs-2/data.yaml'), PosixPath('/content/UAVs-2/valid'), PosixPath('/content/UAVs-2/test'), PosixPath('/content/UAVs-2/train'), PosixPath('/content/UAVs-2/README.roboflow.txt')]


In [8]:
for split in ["train", "valid", "test"]:
    image_dir = BASE / split / "images"
    label_dir = BASE / split / "labels"

    images = len(list(image_dir.glob("*")))
    labels = len(list(label_dir.glob("*")))

    print(f"{split}:")
    print(f"  Images: {images}")
    print(f"  Labels: {labels}")

train:
  Images: 6928
  Labels: 6928
valid:
  Images: 1884
  Labels: 1884
test:
  Images: 450
  Labels: 450


In [9]:
print((BASE / "data.yaml").read_text())

names:
- drone
nc: 1
roboflow:
  license: CC BY 4.0
  project: uavs-vqpqt
  url: https://universe.roboflow.com/uavs-7l7kv/uavs-vqpqt/dataset/2
  version: 2
  workspace: uavs-7l7kv
test: ../test/images
train: ../train/images
val: ../valid/images



## **Train YOLOv8n**

In [10]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

In [ ]:
results = model.train(
    data = f"{BASE}/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project=f"{PROJECT_DIR}/training",
    name="yolov8n_baseline",
    exist_ok=True
)

Ultralytics 8.4.124 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/UAVs-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_baseline, nbs=64, nms=False,

In [11]:
print("Searching for best.pt...")

matches = list(Path("/content").rglob("best.pt"))
for path in matches:
    print(path)

Searching for best.pt...
/content/drive/MyDrive/drone_detection_project/training/yolov8n_baseline/weights/best.pt


In [12]:
best_model = matches[0]
final_model_path = f"{PROJECT_DIR}/models/drone_yolov8n_best.pt"

import shutil

shutil.copy2(best_model, final_model_path)
print("Model saved permanently at:")
print(final_model_path)

Model saved permanently at:
/content/drive/MyDrive/drone_detection_project/models/drone_yolov8n_best.pt


In [13]:
import os

print(
    "Model size:",
    round(os.path.getsize(final_model_path) / (1024 * 1024), 2),
    "MB"
)

Model size: 5.96 MB


## **Final TEST Evaluation**

In [14]:
best_model = YOLO(final_model_path)

test_results = best_model.val(
    data=f"{BASE}/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    project=f"{PROJECT_DIR}/results",
    name="test_evaluation",
    exist_ok=True
)

Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 36.9±26.6 MB/s, size: 59.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/UAVs-2/test/labels.cache... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 69.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 4.5it/s 6.5s
                   all        450        455      0.951       0.94      0.971      0.714
Speed: 3.1ms preprocess, 4.3ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /content/drive/MyDrive/drone_detection_project/results/test_evaluation


## **Generate Detection Images**

In [15]:
prediction_results = best_model.predict(
    source=f"{BASE}/test/images",
    imgsz=640,
    conf=0.25,
    save=True,
    project=f"{PROJECT_DIR}/results",
    name="test_predictions",
    exist_ok=True
)


image 1/450 /content/UAVs-2/test/images/0238_jpg.rf.3c7f1a911859edb2f045a1c271eae0f3.jpg: 640x640 1 drone, 10.3ms
image 2/450 /content/UAVs-2/test/images/0279_jpg.rf.dd3f4ad5da13131f2fc80e07f58b39cd.jpg: 640x640 1 drone, 7.2ms
image 3/450 /content/UAVs-2/test/images/030_jpg.rf.7fb64f102c593207c1a2298212c0d6ab.jpg: 640x640 1 drone, 7.2ms
image 4/450 /content/UAVs-2/test/images/0326_jpg.rf.2364186fe7afd1eef3a044949ce6c188.jpg: 640x640 1 drone, 7.2ms
image 5/450 /content/UAVs-2/test/images/042_jpg.rf.54abe1883821693ed2350b5560efe9af.jpg: 640x640 1 drone, 7.2ms
image 6/450 /content/UAVs-2/test/images/0448_jpg.rf.870d8f68f7b48f4c6c824c3a4dddd724.jpg: 640x640 1 drone, 9.0ms
image 7/450 /content/UAVs-2/test/images/0466_jpg.rf.332c5615f35bbf64557c92b3675aae18.jpg: 640x640 1 drone, 7.2ms
image 8/450 /content/UAVs-2/test/images/047_jpg.rf.6e8a98c2e708fe1ecd2a79a1b7d9c9db.jpg: 640x640 1 drone, 7.2ms
image 9/450 /content/UAVs-2/test/images/047_jpg.rf.e5c483c0258b474a8b2ecfd52cbb2b7c.jpg: 640x640 

## **Video Detection**

In [16]:
video_path = "/content/drone_testing.mp4"

detection_results = best_model.predict(
    source=video_path,
    imgsz=640,
    conf=0.25,
    save=True,
    project=f"{PROJECT_DIR}/videos",
    name="detection",
    exist_ok=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/356) /content/drone_testing.mp4: 640x384 2 drones, 41.4ms
video 1/1 (frame 2/356) /content/drone_testing.mp4: 640x384 2 drones, 15.6ms
video 1/1 (frame 3/356) /content/drone_testing.mp4: 640x384 2 drones, 10.9ms
video 1/1 (frame 4/356) /content/drone_testing.mp4: 640x384 3 drones, 9.7ms
video 1/1 (frame 5/356) /content/drone_testing.mp4: 640x384 2 drones, 9.9ms
video 1/1 (frame 6/356) /content/drone_testing.mp4: 640x384 1 drone, 10.0ms


## **Multi-Drone Tracking**

In [17]:
from pathlib import Path

tracker_config = """
tracker_type: bytetrack
track_high_thresh: 0.20
track_low_thresh: 0.05
new_track_thresh: 0.20
track_buffer: 120
match_thresh: 0.95
fuse_score: True
"""

tracker_path = Path("/content/custom_bytetrack.yaml")
tracker_path.write_text(tracker_config)

print("Custom tracker saved to:", tracker_path)

Custom tracker saved to: /content/custom_bytetrack.yaml


In [18]:
tracking_results = best_model.track(
    source=video_path,
    imgsz=640,
    conf=0.25,
    tracker="/content/custom_bytetrack.yaml",
    persist=True,
    save=True,
    project=f"{PROJECT_DIR}/videos",
    name="tracking",
    exist_ok=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/356) /content/drone_testing.mp4: 640x384 2 drones, 10.4ms
video 1/1 (frame 2/356) /content/drone_testing.mp4: 640x384 2 drones, 10.6ms
video 1/1 (frame 3/356) /content/drone_testing.mp4: 640x384 2 drones, 9.9ms
video 1/1 (frame 4/356) /content/drone_testing.mp4: 640x384 2 drones, 9.9ms
video 1/1 (frame 5/356) /content/drone_testing.mp4: 640x384 2 drones, 10.4ms
video 1/1 (frame 6/356) /content/drone_testing.mp4: 640x384 1 drone, 10.1ms


In [19]:
import time
import cv2

cap = cv2.VideoCapture(video_path)

frame_count = 0
start_time = time.time()

while True:
    ret, frame = cap.read()

    if not ret:
        break

    best_model.predict(
        source=frame,
        imgsz=640,
        conf=0.25,
        verbose=False
    )

    frame_count += 1

cap.release()

elapsed = time.time() - start_time
fps = frame_count / elapsed

print("Frames processed:", frame_count)
print("Elapsed time:", round(elapsed, 2), "seconds")
print("Approximate FPS:", round(fps, 2))

Frames processed: 356
Elapsed time: 12.23 seconds
Approximate FPS: 29.1


In [20]:
# ================================
# RECOVER AFTER COLAB RESTART
# ================================

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
from pathlib import Path
import os

# Install/load Ultralytics if necessary
!pip install -q ultralytics

# Permanent project directory
PROJECT_DIR = "/content/drive/MyDrive/drone_detection_project"

# --------------------------------
# 1. Find the saved model
# --------------------------------

model_path = f"{PROJECT_DIR}/models/drone_yolov8n_best.pt"

print("Model exists:", os.path.exists(model_path))

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Could not find trained model at:\n{model_path}"
    )

# --------------------------------
# 2. Load trained model
# --------------------------------

best_model = YOLO(model_path)

print("Model loaded successfully.")

# --------------------------------
# 3. Locate dataset
# --------------------------------

dataset_candidates = [
    "/content/UAVs-2",
    "/content/UAVs-2-2"
]

dataset_path = None

for path in dataset_candidates:
    if os.path.exists(f"{path}/data.yaml"):
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError(
        "Dataset is not currently available in this Colab runtime."
    )

print("Dataset:", dataset_path)
print("data.yaml exists:", os.path.exists(f"{dataset_path}/data.yaml"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model exists: True
Model loaded successfully.
Dataset: /content/UAVs-2
data.yaml exists: True


In [21]:
test_results = best_model.val(
    data=f"{dataset_path}/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    project=f"{PROJECT_DIR}/results",
    name="final_test_evaluation",
    exist_ok=True
)

Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1073.9±385.4 MB/s, size: 43.3 KB)
val: Scanning /content/UAVs-2/test/labels.cache... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 118.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.8it/s 10.3s
                   all        450        455      0.951       0.94      0.971      0.714
Speed: 4.1ms preprocess, 4.7ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to /content/drive/MyDrive/drone_detection_project/results/final_test_evaluation


In [22]:
print("FINAL TEST RESULTS")
print("==================")

print(f"Precision : {test_results.box.mp:.4f}")
print(f"Recall    : {test_results.box.mr:.4f}")
print(f"mAP@50    : {test_results.box.map50:.4f}")
print(f"mAP@50-95 : {test_results.box.map:.4f}")

FINAL TEST RESULTS
Precision : 0.9511
Recall    : 0.9401
mAP@50    : 0.9709
mAP@50-95 : 0.7139
